# Week 2, Day 2: chunking and embeddings

Two ideas: how you **cut** documents changes what search can find, and an **embedding** turns text into numbers so similar texts get similar numbers. The project compared two chunking strategies (paragraph, fixed window) on 10 to 15 questions, using hashed-word embeddings of 256 numbers. Here is the same experiment in plain Python, no downloads.

Run the cells from the top, one at a time (Shift+Enter).

## Step 1: two chunking strategies
*Paragraph*: one chunk per paragraph. *Fixed window*: windows of a set size that start and end at word boundaries, with overlap (so a fact on a boundary stays whole).

In [ ]:
DOCS = {
 "Leave policy":      "Every full-time employee gets 24 days of paid annual leave per calendar year.\n\nUnused leave carries over up to 10 days and expires on 31 March.\n\nSick leave is separate from annual leave.",
 "Remote work and security": "Employees may work remotely up to 3 days a week after probation.\n\nCompany laptops must use full disk encryption and lock automatically after 5 minutes.\n\nUse the VPN on any public Wi-Fi.",
 "Expenses":          "Business expenses are reimbursed within 14 days of a complete claim.\n\nAny single purchase above 500 euros needs written approval from a director before you buy.",
 "Onboarding and training": "Each employee has a learning budget of 1,500 euros per year for courses, books and conferences.\n\nNew hires are paired with a mentor for their first 3 months.",
}

def paragraph_chunks(body, max_len=420):
    return [p for p in body.split("\n\n") if p.strip()]

def fixed_chunks(body, size=120, overlap=30):
    words, out, i = body.replace("\n\n", " ").split(), [], 0
    while i < len(words):
        j, piece = i, ""
        while j < len(words) and len(piece) + len(words[j]) + 1 <= size: piece += words[j] + " "; j += 1
        out.append(piece.strip());
        if j >= len(words): break
        back = 0                                   # step back so neighbours share about `overlap` characters
        while back < j - i - 1 and len(" ".join(words[j - back - 1:j])) < overlap: back += 1
        i = max(j - back, i + 1)
    return out

def make_chunks(strategy, **kw):
    fn = paragraph_chunks if strategy == "paragraph" else fixed_chunks
    return [{"title": t, "text": c} for t, body in DOCS.items() for c in fn(body, **kw)]

for s in ("paragraph", "fixed"):
    cs = make_chunks(s)
    print(f"{s:10} {len(cs)} chunks, longest {max(len(c['text']) for c in cs)} characters")

## Step 2: embeddings by hashing words
Each word is hashed to one of 256 slots and counted; the vector is scaled to length 1. Texts that share words get close vectors. It is a simple, honest stand-in for a model embedding, and it is the kind the project used on Day 2.

In [ ]:
import hashlib, math, re
DIM = 256
def embed(text):
    v = [0.0] * DIM
    for w in re.findall(r"[a-z0-9]+", text.lower()):
        v[int(hashlib.md5(w.encode()).hexdigest(), 16) % DIM] += 1.0
    norm = math.sqrt(sum(x * x for x in v)) or 1.0
    return [x / norm for x in v]
def cosine(a, b): return sum(x * y for x, y in zip(a, b))      # vectors have length 1, so this is cosine similarity

print("close:    ", round(cosine(embed("laptops lock after 5 minutes"), embed("laptop lock automatically after 5 minutes")), 3))
print("different:", round(cosine(embed("laptops lock after 5 minutes"), embed("learning budget for courses")), 3))

## Step 3: compare the strategies with fixed questions
Each question names the document that holds the answer. For each strategy we ask: is a chunk of the right document in the top 3, and how high is it (MRR, 1.0 means first)?

In [ ]:
QUESTIONS = [
  ("After how many minutes does a laptop lock itself?", "Remote work and security"),
  ("How many days of annual leave do I get?", "Leave policy"),
  ("When does unused leave expire?", "Leave policy"),
  ("Who approves a purchase above 500 euros?", "Expenses"),
  ("How soon are claims reimbursed?", "Expenses"),
  ("What is the yearly learning budget?", "Onboarding and training"),
  ("How long does a new hire have a mentor?", "Onboarding and training"),
  ("How many remote days a week are allowed?", "Remote work and security"),
]
def evaluate(chunks, k=3):
    vecs = [embed(c["text"]) for c in chunks]
    hits, rr = 0, 0.0
    for q, doc in QUESTIONS:
        qv = embed(q)
        order = sorted(range(len(chunks)), key=lambda i: -cosine(qv, vecs[i]))[:k]
        ranks = [r + 1 for r, i in enumerate(order) if chunks[i]["title"] == doc]
        if ranks: hits += 1; rr += 1 / ranks[0]
    return round(100 * hits / len(QUESTIONS), 1), round(rr / len(QUESTIONS), 3)

print("strategy               chunks  hit%   MRR")
for label, cs in (("paragraph", make_chunks("paragraph")), ("fixed 120/30", make_chunks("fixed", size=120, overlap=30)), ("fixed 60/15", make_chunks("fixed", size=60, overlap=15)), ("fixed 250/60", make_chunks("fixed", size=250, overlap=60))):
    h, m = evaluate(cs)
    print(f"{label:22} {len(cs):5}  {h:5}  {m}")

## Step 4: practise
1. Which strategy and size scored best here? The project chose by the same kind of table, then explained why.
2. Add 3 questions of your own and re-run. Does the winner change? (With only 8 questions, one question is 12.5 points: treat small gaps as noise.)
3. Change `DIM` to 64. What happens to the scores, and why? (Hint: more words share a slot.)

**Where this is in the project:** `RagEngine.cs` (both chunkers, the hashed embeddings), `docs/evidence/week2-day2.md`.